# Rocket Lab - Atividade 1
<br>

## Bronze para Silver

Segundo notebook da pipeline da atividade, com os seguintes objetivos:
- ler os dados brutos da camada Bronze;
- aplicar regras de limpeza, padronização e tipagem;
- renomear colunas para português, conforme as regras de negócio;
- criar as tabelas tratadas da camada Silver no formato Delta.

Nesta etapa, os dados passam por padronização de tipos, tratamento de nulos e demais ajustes necessários para consumo analítico.
___

#### Etapa de Imports, definições que o notebook vai usar e criação e configurações da camada Silver
___


In [0]:
# Célula somente para imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DateType, TimestampType, IntegerType, DoubleType, StringType


In [0]:
# Criação schema da camada Silver
spark.sql("CREATE SCHEMA IF NOT EXISTS ecommerce_olist.silver")

spark.sql("SHOW CATALOGS").show()
spark.sql("SHOW SCHEMAS IN ecommerce_olist").show()

+---------------+
|        catalog|
+---------------+
|ecommerce_olist|
|        samples|
|         system|
|      workspace|
+---------------+

+------------------+
|      databaseName|
+------------------+
|            bronze|
|           default|
|              gold|
|information_schema|
|           landing|
|            silver|
+------------------+



In [0]:
# Célula apenas para confirmar a utilização do catálogo e schema certos
spark.sql("USE CATALOG ecommerce_olist")
spark.sql("USE SCHEMA silver")

DataFrame[]

In [0]:
# Definições dos caminhos e leitura das tabelas bronze para dfs
volume_path = "/Volumes/ecommerce_olist/landing/dados_olist"

catalogo = "ecommerce_olist"
schema = "silver"

df_customers = spark.table("bronze.tb_customers")
df_orders = spark.table("bronze.tb_orders")
df_order_items = spark.table("bronze.tb_order_items")
df_order_payments = spark.table("bronze.tb_order_payments")
df_order_reviews = spark.table("bronze.tb_order_reviews")
df_products = spark.table("bronze.tb_products")
df_sellers = spark.table("bronze.tb_sellers")
df_category_translation = spark.table("bronze.tb_product_category_name_translation")
df_cotacao_dolar = spark.table("bronze.tb_cotacao_dolar")

#### Etapa de tratamento, padronização e estruturação das tabelas da camada Silver

Nesta etapa, cada tabela será tratada em uma seção específica

OBS.: As colunas de "timestamp_ingestion" serão utilizadas para processamentos e removidas logo após
___

##### dim_consumidores

In [0]:
# Visualização inicial
display(df_customers.limit(5))
df_customers.printSchema()

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_name,customer_gender,customer_birth_date,customer_age,timestamp_ingestion
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,Zoe Nogueira,F,1956-09-07,70,2026-04-07T20:24:18.288Z
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,Liam Viana,M,1974-09-23,52,2026-04-07T20:24:18.288Z
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,Diego Aparecida,M,1964-05-20,62,2026-04-07T20:24:18.288Z
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,Marcos Vinicius Azevedo,M,1967-01-14,59,2026-04-07T20:24:18.288Z
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,Srta. Juliana Siqueira,F,1950-08-01,76,2026-04-07T20:24:18.288Z


root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_gender: string (nullable = true)
 |-- customer_birth_date: string (nullable = true)
 |-- customer_age: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
# Deduplicação dos consumidores com Window Function
window_consumidores = Window.partitionBy("customer_id").orderBy(F.col("timestamp_ingestion").desc())

df_customers_deduplicado = (df_customers.withColumn("rn", F.row_number().over(window_consumidores)).filter(F.col("rn") == 1).drop("rn"))

# Verificação final de duplicidade de customer_id
display(df_customers_deduplicado.groupBy("customer_id").count().filter(F.col("count") > 1))

customer_id,count


In [0]:
# Tradução dos nomes das colunas
df_consumidores_renomeado = (
    df_customers_deduplicado
    .select(
        F.col("customer_id").alias("id_consumidor"),
        F.col("customer_unique_id").alias("id_unico_consumidor"),
        F.col("customer_zip_code_prefix").alias("prefixo_cep"),
        F.col("customer_name").alias("nome_consumidor"),
        F.col("customer_gender").alias("genero_consumidor"),
        F.col("customer_birth_date").alias("data_nascimento"),
        F.col("customer_age").alias("idade_consumidor"),
        F.col("customer_city").alias("cidade"),
        F.col("customer_state").alias("estado"),
    )
)

In [0]:
# Tratamento dos tipos e padronização das strings
df_dim_consumidores = (
    df_consumidores_renomeado
    .withColumn("id_consumidor", F.trim(F.col("id_consumidor")))
    .withColumn("id_unico_consumidor", F.trim(F.col("id_unico_consumidor")))
    .withColumn("prefixo_cep", F.col("prefixo_cep").cast("string"))
    .withColumn("nome_consumidor", F.trim(F.col("nome_consumidor")))
    .withColumn("genero_consumidor", F.trim(F.col("genero_consumidor")))
    .withColumn("data_nascimento", F.to_date("data_nascimento"))
    .withColumn("idade_consumidor", F.col("idade_consumidor").cast("int"))
    .withColumn("cidade", F.upper(F.trim(F.col("cidade"))))
    .withColumn("estado", F.upper(F.trim(F.col("estado"))))
)

In [0]:
# Conversão de strings vazias para nulo em colunas de texto
df_dim_consumidores = (
    df_dim_consumidores
    .withColumn("nome_consumidor", F.when(F.col("nome_consumidor") == "", None).otherwise(F.col("nome_consumidor")))
    .withColumn("genero_consumidor", F.when(F.col("genero_consumidor") == "", None).otherwise(F.col("genero_consumidor")))
    .withColumn("cidade", F.when(F.col("cidade") == "", None).otherwise(F.col("cidade")))
    .withColumn("estado", F.when(F.col("estado") == "", None).otherwise(F.col("estado")))
)

In [0]:
# Verificação de nulos
df_dim_consumidores.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_dim_consumidores.columns
]).display()

id_consumidor,id_unico_consumidor,prefixo_cep,nome_consumidor,genero_consumidor,data_nascimento,idade_consumidor,cidade,estado
0,0,0,0,0,0,0,0,0


In [0]:
# Tratamento de nulos em colunas de texto
df_dim_consumidores = (
    df_dim_consumidores
    .fillna({
        "nome_consumidor": "NAO INFORMADO",
        "genero_consumidor": "NAO INFORMADO",
        "cidade": "NAO INFORMADO",
        "estado": "NAO INFORMADO"
    })
)

# OBS.: Cep não está sendo tratado, pois apesar de ser uma string, utiliza números como valores dos dados, então acredito que colocar um "NAO INFORMADO" pode gerar confusão

In [0]:
# Visualização final
display(df_dim_consumidores.limit(10))
df_dim_consumidores.printSchema()

id_consumidor,id_unico_consumidor,prefixo_cep,nome_consumidor,genero_consumidor,data_nascimento,idade_consumidor,cidade,estado
00012a2ce6f8dcda20d059ce98491703,248ffe10d632bebe4f7267f1f44844c9,6273,Dr. Davi Pinto,M,1971-07-03,55,OSASCO,SP
000161a058600d5901f007fab4c27140,b0015e09bb4b6e47c52844fab5fb6638,35550,Sr. Ravi Lucca Sousa,M,1991-03-07,35,ITAPECERICA,MG
0001fd6190edaaf884bcaf3d49edf079,94b11d37cd61cb2994a194d11f89682b,29830,Giovanna Ramos,F,1952-02-19,74,NOVA VENECIA,ES
0002414f95344307404f0ace7a26f1d5,4893ad4ea28b2c5b3ddf4e82e79db9e6,39664,Lívia Nogueira,F,2008-02-05,18,MENDONCA,MG
000379cdec625522490c315e70c7a9fb,0b83f73b19c2019e182fd552c048a22c,4841,Srta. Maria Laura Moura,F,1952-06-29,74,SAO PAULO,SP
0004164d20a9e969af783496f3408652,104bdb7e6a6cdceaa88c3ea5fa6b2b93,13272,Sr. Rafael Cavalcanti,M,1965-03-25,61,VALINHOS,SP
000419c5494106c306a97b5635748086,14843983d4a159080f6afe4b7f346e7c,24220,Luiz Otávio Costa,M,1988-02-16,38,NITEROI,RJ
00046a560d407e99b969756e0b10f282,0b5295fc9819d831f68eb0e9a3e13ab7,20540,Alana Castro,F,1972-02-18,54,RIO DE JANEIRO,RJ
00050bf6e01e69d5c0fd612f1bcfb69c,e3cf594a99e810f58af53ed4820f25e5,98700,Luna Rocha,F,1997-11-07,29,IJUI,RS
000598caf2ef4117407665ac33275130,7e0516b486e92ed3f3afdd6d1276cfbd,35540,Emanuelly Araújo,F,1951-10-21,75,OLIVEIRA,MG


root
 |-- id_consumidor: string (nullable = true)
 |-- id_unico_consumidor: string (nullable = true)
 |-- prefixo_cep: string (nullable = true)
 |-- nome_consumidor: string (nullable = false)
 |-- genero_consumidor: string (nullable = false)
 |-- data_nascimento: date (nullable = true)
 |-- idade_consumidor: integer (nullable = true)
 |-- cidade: string (nullable = false)
 |-- estado: string (nullable = false)



In [0]:
# Salva a tabela
df_dim_consumidores.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{schema}.dim_consumidores")


##### fat_pedidos

In [0]:
# Visualização inicial
display(df_orders.limit(5))
df_orders.printSchema()

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,timestamp_ingestion
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,2026-04-07T20:24:36.062Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,2026-04-07T20:24:36.062Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,2026-04-07T20:24:36.062Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,2026-04-07T20:24:36.062Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,2026-04-07T20:24:36.062Z


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
# Por mais que não tenha valores duplicados e não estar descrito como requisito, decidi adicionar deduplicação nesse caso também. (Levando em consideração possíveis novas ingestões de dados)
window_pedidos = Window.partitionBy("order_id").orderBy(F.col("timestamp_ingestion").desc())

df_orders_deduplicado = df_orders.withColumn("rn", F.row_number().over(window_pedidos)).filter(F.col("rn") == 1).drop("rn")

# Apenas visualização do df
display(df_orders_deduplicado.groupBy("order_id").count().filter(F.col("count") > 1))

order_id,count


In [0]:
# Tradução dos nomes das colunas
df_pedidos_renomeado = (
    df_orders_deduplicado
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("customer_id").alias("id_consumidor"),
        F.col("order_status").alias("status"),
        F.col("order_purchase_timestamp").alias("data_compra"),
        F.col("order_approved_at").alias("data_aprovacao"),
        F.col("order_delivered_carrier_date").alias("data_envio_transportadora"),
        F.col("order_delivered_customer_date").alias("data_entrega"),
        F.col("order_estimated_delivery_date").alias("data_entrega_estimada")
    )
)

In [0]:
# Tratamento dos tipos e padronização das colunas de texto
df_pedidos_tratado = (
    df_pedidos_renomeado
    .withColumn("id_pedido", F.trim(F.col("id_pedido")))
    .withColumn("id_consumidor", F.trim(F.col("id_consumidor")))
    .withColumn("status", F.trim(F.lower(F.col("status"))))
    .withColumn("data_compra", F.to_timestamp("data_compra"))
    .withColumn("data_aprovacao", F.to_timestamp("data_aprovacao"))
    .withColumn("data_envio_transportadora", F.to_timestamp("data_envio_transportadora"))
    .withColumn("data_entrega", F.to_timestamp("data_entrega"))
    .withColumn("data_entrega_estimada", F.to_timestamp("data_entrega_estimada"))
)

In [0]:
# Mapeamento e tradução dos valores de status
status_map = F.create_map(
    F.lit("delivered"), F.lit("entregue"),
    F.lit("canceled"), F.lit("cancelado"),
    F.lit("shipped"), F.lit("enviado"),
    F.lit("processing"), F.lit("processando"),
    F.lit("invoiced"), F.lit("faturado"),
    F.lit("unavailable"), F.lit("indisponível"),
    F.lit("created"), F.lit("criado"),
    F.lit("approved"), F.lit("aprovado")
)

df_fat_pedidos = df_pedidos_tratado.withColumn("status", status_map[F.col("status")])


In [0]:
# Identificação de valores nulos por coluna
display(
    df_fat_pedidos.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df_fat_pedidos.columns
    ])
)

id_pedido,id_consumidor,status,data_compra,data_aprovacao,data_envio_transportadora,data_entrega,data_entrega_estimada
0,0,0,0,160,1783,2965,0


In [0]:
# Tratamento de nulos textuais (Creio que a única coluna que tem um certo impacto em ter um valor nulo é a de status, uma vez que as outras de data, por exemplo, podem estar nulas por não ter chegado em determinada etapa do processo, exemplo: se um produto não foi enviado ele não deve ter data de envio)
df_fat_pedidos = (
    df_fat_pedidos
    .fillna({
        "status": "NAO INFORMADO",
    })
)

In [0]:
# Visualização das colunas originais após o tratamento
display(df_fat_pedidos.limit(5))

id_pedido,id_consumidor,status,data_compra,data_aprovacao,data_envio_transportadora,data_entrega,data_entrega_estimada
00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,entregue,2017-09-13T08:59:02.000Z,2017-09-13T09:45:35.000Z,2017-09-19T18:34:16.000Z,2017-09-20T23:43:48.000Z,2017-09-29T00:00:00.000Z
00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,entregue,2017-04-26T10:53:06.000Z,2017-04-26T11:05:13.000Z,2017-05-04T14:35:00.000Z,2017-05-12T16:04:24.000Z,2017-05-15T00:00:00.000Z
000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,entregue,2018-01-14T14:33:31.000Z,2018-01-14T14:48:30.000Z,2018-01-16T12:36:48.000Z,2018-01-22T13:19:16.000Z,2018-02-05T00:00:00.000Z
00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,entregue,2018-08-08T10:00:35.000Z,2018-08-08T10:10:18.000Z,2018-08-10T13:28:00.000Z,2018-08-14T13:32:39.000Z,2018-08-20T00:00:00.000Z
00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,entregue,2017-02-04T13:57:51.000Z,2017-02-04T14:10:13.000Z,2017-02-16T09:46:09.000Z,2017-03-01T16:42:31.000Z,2017-03-17T00:00:00.000Z


In [0]:
# Criação das colunas derivadas
df_fat_pedidos = (
    df_fat_pedidos
    .withColumn(
        "tempo_entrega_dias",
        F.datediff(F.col("data_entrega"), F.col("data_compra"))
    )
    .withColumn(
        "tempo_entrega_estimado_dias",
        F.datediff(F.col("data_entrega_estimada"), F.col("data_compra"))
    )
    .withColumn(
        "diferenca_entrega_dias",
        F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias")
    )
    .withColumn(
        "entrega_no_prazo",
        F.when(F.col("data_entrega").isNull(), F.lit("Não Entregue"))
         .when(F.col("diferenca_entrega_dias") <= 0, F.lit("Sim"))
         .otherwise(F.lit("Não"))
    )
)

In [0]:
# Visualização final do df
display(df_fat_pedidos.limit(10))

id_pedido,id_consumidor,status,data_compra,data_aprovacao,data_envio_transportadora,data_entrega,data_entrega_estimada,tempo_entrega_dias,tempo_entrega_estimado_dias,diferenca_entrega_dias,entrega_no_prazo
00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,entregue,2017-09-13T08:59:02.000Z,2017-09-13T09:45:35.000Z,2017-09-19T18:34:16.000Z,2017-09-20T23:43:48.000Z,2017-09-29T00:00:00.000Z,7,16,-9,Sim
00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,entregue,2017-04-26T10:53:06.000Z,2017-04-26T11:05:13.000Z,2017-05-04T14:35:00.000Z,2017-05-12T16:04:24.000Z,2017-05-15T00:00:00.000Z,16,19,-3,Sim
000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,entregue,2018-01-14T14:33:31.000Z,2018-01-14T14:48:30.000Z,2018-01-16T12:36:48.000Z,2018-01-22T13:19:16.000Z,2018-02-05T00:00:00.000Z,8,22,-14,Sim
00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,entregue,2018-08-08T10:00:35.000Z,2018-08-08T10:10:18.000Z,2018-08-10T13:28:00.000Z,2018-08-14T13:32:39.000Z,2018-08-20T00:00:00.000Z,6,12,-6,Sim
00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,entregue,2017-02-04T13:57:51.000Z,2017-02-04T14:10:13.000Z,2017-02-16T09:46:09.000Z,2017-03-01T16:42:31.000Z,2017-03-17T00:00:00.000Z,25,41,-16,Sim
00048cc3ae777c65dbb7d2a0634bc1ea,816cbea969fe5b689b39cfc97a506742,entregue,2017-05-15T21:42:34.000Z,2017-05-17T03:55:27.000Z,2017-05-17T11:05:55.000Z,2017-05-22T13:44:35.000Z,2017-06-06T00:00:00.000Z,7,22,-15,Sim
00054e8431b9d7675808bcb819fb4a32,32e2e6ab09e778d99bf2e0ecd4898718,entregue,2017-12-10T11:53:48.000Z,2017-12-10T12:10:31.000Z,2017-12-12T01:07:48.000Z,2017-12-18T22:03:38.000Z,2018-01-04T00:00:00.000Z,8,25,-17,Sim
000576fe39319847cbb9d288c5617fa6,9ed5e522dd9dd85b4af4a077526d8117,entregue,2018-07-04T12:08:27.000Z,2018-07-05T16:35:48.000Z,2018-07-05T12:15:00.000Z,2018-07-09T14:04:07.000Z,2018-07-25T00:00:00.000Z,5,21,-16,Sim
0005a1a1728c9d785b8e2b08b904576c,16150771dfd4776261284213b89c304e,entregue,2018-03-19T18:40:33.000Z,2018-03-20T18:35:21.000Z,2018-03-28T00:37:42.000Z,2018-03-29T18:17:31.000Z,2018-03-29T00:00:00.000Z,10,10,0,Sim
0005f50442cb953dcd1d21e1fb923495,351d3cb2cee3c7fd0af6616c82df21d3,entregue,2018-07-02T13:59:39.000Z,2018-07-02T14:10:56.000Z,2018-07-03T14:25:00.000Z,2018-07-04T17:28:31.000Z,2018-07-23T00:00:00.000Z,2,21,-19,Sim


In [0]:
# Salva a tabela
df_fat_pedidos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{schema}.fat_pedidos")

##### fat_itens_pedidos

In [0]:
# Visualização inicial
display(df_order_items.limit(5))
df_order_items.printSchema()

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,timestamp_ingestion
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,2026-04-07T20:24:27.126Z
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,2026-04-07T20:24:27.126Z
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,2026-04-07T20:24:27.126Z
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,2026-04-07T20:24:27.126Z
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,2026-04-07T20:24:27.126Z


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- freight_value: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
# Deduplicação dos itens de pedido.
# Como um pedido pode ter vários itens (e continua tendo o mesmo order_id), a busca dessa vez é feita a partir do order_id + order_item_id.
window_itens_pedidos = Window.partitionBy("order_id", "order_item_id").orderBy(F.col("timestamp_ingestion").desc())

df_order_items_deduplicado = (df_order_items.withColumn("rn", F.row_number().over(window_itens_pedidos)).filter(F.col("rn") == 1).drop("rn"))


# Verificação final de duplicidade da de "order_id" + "order_item_id"
display(df_order_items_deduplicado.groupBy("order_id", "order_item_id").count().filter(F.col("count") > 1))

order_id,order_item_id,count


In [0]:
# Tradução dos nomes das colunas
df_itens_pedidos_renomeado = (
    df_order_items_deduplicado
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("order_item_id").alias("id_item"),
        F.col("product_id").alias("id_produto"),
        F.col("seller_id").alias("id_vendedor"),
        F.col("shipping_limit_date").alias("data_limite_envio"),
        F.col("price").alias("preco_BRL"),
        F.col("freight_value").alias("preco_frete")
    )
)

In [0]:
# Tratamento dos tipos e padronização das strings
df_fat_itens_pedidos = (
    df_itens_pedidos_renomeado
    .withColumn("id_pedido", F.trim(F.col("id_pedido")))
    .withColumn("id_item", F.col("id_item").cast("int"))
    .withColumn("id_produto", F.trim(F.col("id_produto")))
    .withColumn("id_vendedor", F.trim(F.col("id_vendedor")))
    .withColumn("data_limite_envio", F.to_timestamp("data_limite_envio"))
    .withColumn("preco_BRL", F.col("preco_BRL").cast("decimal(10,2)"))
    .withColumn("preco_frete", F.col("preco_frete").cast("decimal(10,2)"))
)

In [0]:
# Conversão de strings vazias em nulos nas colunas textuais
df_fat_itens_pedidos = (
    df_fat_itens_pedidos
    .withColumn("id_pedido",F.when(F.col("id_pedido") == "", None).otherwise(F.col("id_pedido")))
    .withColumn("id_produto",F.when(F.col("id_produto") == "", None).otherwise(F.col("id_produto")))
    .withColumn("id_vendedor",F.when(F.col("id_vendedor") == "", None).otherwise(F.col("id_vendedor")))
)

In [0]:
# Verificação de nulos por coluna
display(
    df_fat_itens_pedidos.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df_fat_itens_pedidos.columns
    ])
)

id_pedido,id_item,id_produto,id_vendedor,data_limite_envio,preco_BRL,preco_frete
0,0,0,0,0,0,0


In [0]:
# Sendo sincero, fiquei em dúvida sobre o que fazer com os possíveis valores nulos, mas acredito que não seria uma boa ideia inventar um dado, dar um drop ou simplesmente deixar passar, por isso, optei por uma flag que indica a coluna (em que o dado nulo é problemático) que apresentou um dado nulo
df_fat_itens_pedidos = (
    df_fat_itens_pedidos
    .withColumn(
        "flag_qualidade",
        F.when(
            F.col("id_pedido").isNull() |
            F.col("id_item").isNull() |
            F.col("id_produto").isNull() |
            F.col("id_vendedor").isNull(),
            F.lit("id_essencial_nulo")
        ).when(
            F.col("preco_BRL").isNull(),
            F.lit("preco_nulo")
        ).when(
            F.col("preco_frete").isNull(),
            F.lit("frete_nulo")
        ).otherwise(F.lit(None))
    )
)

In [0]:
# Visualização dos tipos de problema que foram capturados pela coluna de qualidade
display(df_fat_itens_pedidos.groupBy("flag_qualidade").count())

flag_qualidade,count
null,112650


In [0]:
# Visualização final
display(df_fat_itens_pedidos.limit(10))
df_fat_itens_pedidos.printSchema()

id_pedido,id_item,id_produto,id_vendedor,data_limite_envio,preco_BRL,preco_frete,flag_qualidade
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.90,13.29,null
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.90,19.93,null
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.00,17.87,null
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79,null
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.90,18.14,null
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23T03:55:27.000Z,21.90,12.69,null
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14T12:10:31.000Z,19.90,11.85,null
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10T12:30:45.000Z,810.00,70.75,null
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26T18:31:29.000Z,145.95,11.65,null
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06T14:10:56.000Z,53.99,11.40,null


root
 |-- id_pedido: string (nullable = true)
 |-- id_item: integer (nullable = true)
 |-- id_produto: string (nullable = true)
 |-- id_vendedor: string (nullable = true)
 |-- data_limite_envio: timestamp (nullable = true)
 |-- preco_BRL: decimal(10,2) (nullable = true)
 |-- preco_frete: decimal(10,2) (nullable = true)
 |-- flag_qualidade: string (nullable = true)



In [0]:
# Salva a tabela
df_fat_itens_pedidos.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{schema}.fat_itens_pedidos")

##### fat_pagamentos_pedidos

In [0]:
# Visualização inicial
display(df_order_payments.limit(5))
df_order_payments.printSchema()

order_id,payment_sequential,payment_type,payment_installments,payment_value,timestamp_ingestion
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,2026-04-07T20:24:30.060Z
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,2026-04-07T20:24:30.060Z
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,2026-04-07T20:24:30.060Z
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,2026-04-07T20:24:30.060Z
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,2026-04-07T20:24:30.060Z


root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: string (nullable = true)
 |-- payment_value: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
# Deduplicação dos pagamentos do pedido utilizando o id do pedido + a ocorrência do pagamento
window_pagamentos_pedidos = Window.partitionBy("order_id", "payment_sequential").orderBy(F.col("timestamp_ingestion").desc())

df_order_payments_deduplicado = df_order_payments.withColumn("rn", F.row_number().over(window_pagamentos_pedidos)).filter(F.col("rn") == 1).drop("rn")


# Verificação final de duplicidade
display(df_order_payments_deduplicado.groupBy("order_id", "payment_sequential").count().filter(F.col("count") > 1))

order_id,payment_sequential,count


In [0]:
# Tradução dos nomes das colunas
df_pagamentos_pedidos_renomeado = (
    df_order_payments_deduplicado
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("payment_sequential").alias("sequencial_pagamento"),
        F.col("payment_type").alias("tipo_pagamento"),
        F.col("payment_installments").alias("quantidade_parcelas"),
        F.col("payment_value").alias("valor_pagamento_BRL")
    )
)

In [0]:
# Tratamento dos tipos e padronização das strings
df_fat_pagamentos_pedidos = (
    df_pagamentos_pedidos_renomeado
    .withColumn("id_pedido", F.trim(F.col("id_pedido")))
    .withColumn("sequencial_pagamento", F.col("sequencial_pagamento").cast("int"))
    .withColumn("tipo_pagamento", F.trim(F.col("tipo_pagamento")))
    .withColumn("quantidade_parcelas", F.col("quantidade_parcelas").cast("int"))
    .withColumn("valor_pagamento_BRL", F.col("valor_pagamento_BRL").cast("decimal(10,2)"))
)

In [0]:
# Tradução dos valores de tipo_pagamento para português
df_fat_pagamentos_pedidos = (
    df_fat_pagamentos_pedidos
    .withColumn(
        "tipo_pagamento",
        F.when(F.col("tipo_pagamento") == "credit_card", "Cartão de Crédito")
         .when(F.col("tipo_pagamento") == "boleto", "Boleto")
         .when(F.col("tipo_pagamento") == "voucher", "Voucher")
         .when(F.col("tipo_pagamento") == "debit_card", "Cartão de Débito")
         .when(F.col("tipo_pagamento") == "not_defined", "Não Definido")
         .otherwise(F.col("tipo_pagamento"))
    )
)

In [0]:
# Conversão de strings vazias em nulos nas colunas textuais
df_fat_pagamentos_pedidos = (
    df_fat_pagamentos_pedidos
    .withColumn("id_pedido",F.when(F.col("id_pedido") == "", None).otherwise(F.col("id_pedido")))

    .withColumn("tipo_pagamento",F.when(F.col("tipo_pagamento") == "", None).otherwise(F.col("tipo_pagamento")))
)

In [0]:
# Verificação de nulos por coluna
display(
    df_fat_pagamentos_pedidos.select([F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df_fat_pagamentos_pedidos.columns
    ])
)

id_pedido,sequencial_pagamento,tipo_pagamento,quantidade_parcelas,valor_pagamento_BRL
0,0,0,0,0


In [0]:
# Criação de uma flag qualidade
# Novamente fiquei com receio de tratar o dado nulo de uma maneira drástica, portanto, preferi apenas sinalizar caso exista alguma falha
df_fat_pagamentos_pedidos = (
    df_fat_pagamentos_pedidos
    .withColumn(
        "flag_qualidade",
        F.when(
            F.col("id_pedido").isNull() |
            F.col("sequencial_pagamento").isNull(),
            F.lit("id_essencial_nulo")
        ).when(
            F.col("tipo_pagamento").isNull(),
            F.lit("tipo_pagamento_nulo")
        ).when(
            F.col("valor_pagamento_BRL").isNull(),
            F.lit("valor_pagamento_nulo")
        ).otherwise(F.lit(None))
    )
)

In [0]:
# Visualização dos tipos de problema que foram capturados pela coluna de qualidade
display(df_fat_pagamentos_pedidos.groupBy("flag_qualidade").count())

flag_qualidade,count
null,103886


In [0]:
# Visualização final
display(df_fat_pagamentos_pedidos.limit(10))
df_fat_pagamentos_pedidos.printSchema()

id_pedido,sequencial_pagamento,tipo_pagamento,quantidade_parcelas,valor_pagamento_BRL,flag_qualidade
00010242fe8c5a6d1ba2dd792cb16214,1,Cartão de Crédito,2,72.19,null
00018f77f2f0320c557190d7a144bdd3,1,Cartão de Crédito,3,259.83,null
000229ec398224ef6ca0657da4fc703e,1,Cartão de Crédito,5,216.87,null
00024acbcdf0a6daa1e931b038114c75,1,Cartão de Crédito,2,25.78,null
00042b26cf59d7ce69dfabb4e55b4fd9,1,Cartão de Crédito,3,218.04,null
00048cc3ae777c65dbb7d2a0634bc1ea,1,Boleto,1,34.59,null
00054e8431b9d7675808bcb819fb4a32,1,Cartão de Crédito,1,31.75,null
000576fe39319847cbb9d288c5617fa6,1,Cartão de Crédito,10,880.75,null
0005a1a1728c9d785b8e2b08b904576c,1,Cartão de Crédito,3,157.60,null
0005f50442cb953dcd1d21e1fb923495,1,Cartão de Crédito,1,65.39,null


root
 |-- id_pedido: string (nullable = true)
 |-- sequencial_pagamento: integer (nullable = true)
 |-- tipo_pagamento: string (nullable = true)
 |-- quantidade_parcelas: integer (nullable = true)
 |-- valor_pagamento_BRL: decimal(10,2) (nullable = true)
 |-- flag_qualidade: string (nullable = true)



In [0]:
# Salva a tabela
df_fat_pagamentos_pedidos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{schema}.fat_pagamentos_pedidos")

##### fat_avaliacoes_pedidos

In [0]:
# Visualização inicial
display(df_order_reviews.limit(5))
df_order_reviews.printSchema()

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,timestamp_ingestion
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18 00:00:00,2018-01-18 21:46:59,2026-04-07T20:24:32.754Z
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10 00:00:00,2018-03-11 03:05:13,2026-04-07T20:24:32.754Z
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17 00:00:00,2018-02-18 14:36:24,2026-04-07T20:24:32.754Z
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,2026-04-07T20:24:32.754Z
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53,2026-04-07T20:24:32.754Z


root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
# Deduplicação das avaliações
window_avaliacoes = Window.partitionBy("review_id").orderBy(F.col("timestamp_ingestion").desc())

df_order_reviews_deduplicado = (df_order_reviews.withColumn("rn", F.row_number().over(window_avaliacoes)).filter(F.col("rn") == 1).drop("rn"))

# Verificação final de duplicidade
display(df_order_reviews_deduplicado.groupBy("review_id").count().filter(F.col("count") > 1))

review_id,count


In [0]:
# Tradução dos nomes das colunas
df_avaliacoes_pedidos_renomeado = (
    df_order_reviews_deduplicado
    .select(
        F.col("review_id").alias("id_avaliacao"),
        F.col("order_id").alias("id_pedido"),
        F.col("review_score").alias("nota_avaliacao"),
        F.col("review_comment_title").alias("titulo_avaliacao_comentario"),
        F.col("review_comment_message").alias("mensagem_avaliacao_comentario"),
        F.col("review_creation_date").alias("data_criacao_avaliacao"),
        F.col("review_answer_timestamp").alias("data_resposta_avaliacao"),
    )
)

In [0]:
# Tratamento dos tipos e padronização das strings (utilizando o try_to_timestamp)
df_fat_avaliacoes_pedidos = (
    df_avaliacoes_pedidos_renomeado
    .withColumn("id_avaliacao", F.trim(F.col("id_avaliacao")))
    .withColumn("id_pedido", F.trim(F.col("id_pedido")))
    .withColumn("nota_avaliacao", F.col("nota_avaliacao").cast("int"))
    .withColumn("titulo_avaliacao_comentario", F.trim(F.col("titulo_avaliacao_comentario")))
    .withColumn("mensagem_avaliacao_comentario", F.trim(F.col("mensagem_avaliacao_comentario")))
    .withColumn("data_criacao_avaliacao", F.expr("try_to_timestamp(data_criacao_avaliacao)"))
    .withColumn("data_resposta_avaliacao", F.expr("try_to_timestamp(data_resposta_avaliacao)"))
)

In [0]:
# Conversão de strings vazias para nulo nas colunas textuais
df_fat_avaliacoes_pedidos = (
    df_fat_avaliacoes_pedidos
    .withColumn("id_avaliacao", F.when(F.col("id_avaliacao") == "", None).otherwise(F.col("id_avaliacao")))
    .withColumn("id_pedido", F.when(F.col("id_pedido") == "", None).otherwise(F.col("id_pedido")))
    .withColumn(
        "titulo_avaliacao_comentario",
        F.when(F.col("titulo_avaliacao_comentario") == "", None).otherwise(F.col("titulo_avaliacao_comentario"))
    )
    .withColumn(
        "mensagem_avaliacao_comentario",
        F.when(F.col("mensagem_avaliacao_comentario") == "", None).otherwise(F.col("mensagem_avaliacao_comentario"))
    )
)

In [0]:
# Tratamento de nulos em título e comentário
df_fat_avaliacoes_pedidos = (
    df_fat_avaliacoes_pedidos
    .fillna({
        "titulo_avaliacao_comentario": "Sem título",
        "mensagem_avaliacao_comentario": "Sem comentário"
    })
)

In [0]:
# Remoção de registros com pedido inválido

# Seleciona através da tabela de pedidos os ids válidos dos pedidos (que não estão nulos)
df_pedidos_validos = (
    df_orders
    .select(F.trim(F.col("order_id")).alias("id_pedido"))
    .filter(F.col("id_pedido").isNotNull())
    .dropDuplicates()
)

# Seleciona os registros válidos através de um inner join pelos IDs dos pedidos, assim, evitando que avaliações para pedido "inexistentes" sejam mantidas
df_fat_avaliacoes_pedidos = (
    df_fat_avaliacoes_pedidos
    .filter(F.col("id_pedido").isNotNull())
    .join(df_pedidos_validos, on="id_pedido", how="inner")
)

In [0]:
# Remoção de registros com datas de comentário nulas ou no futuro
# Como no documento de requisitos ficou descrito remoção de datas de comentário, fiquei sem saber se as respostas entravam nesse requisito. Por se tratar de uma data, e querendo ou não uma resposta é um "comentário", preferi tratar também.
df_fat_avaliacoes_pedidos = (
    df_fat_avaliacoes_pedidos
    .filter(F.col("data_criacao_avaliacao").isNotNull())
    .filter(F.col("data_criacao_avaliacao") <= F.current_timestamp())
    .filter(
        F.col("data_resposta_avaliacao").isNull() |
        (F.col("data_resposta_avaliacao") <= F.current_timestamp())
    )
)

In [0]:
# Verificação de nulos por coluna
display(df_fat_avaliacoes_pedidos.select([F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df_fat_avaliacoes_pedidos.columns
    ])
)

id_pedido,id_avaliacao,nota_avaliacao,titulo_avaliacao_comentario,mensagem_avaliacao_comentario,data_criacao_avaliacao,data_resposta_avaliacao
0,0,0,0,0,0,0


In [0]:
# Visualização final
display(df_fat_avaliacoes_pedidos.limit(10))
df_fat_avaliacoes_pedidos.printSchema()

id_pedido,id_avaliacao,nota_avaliacao,titulo_avaliacao_comentario,mensagem_avaliacao_comentario,data_criacao_avaliacao,data_resposta_avaliacao
75cc0defdf90cda65066ce0ac171e2ab,0005bc906dfe3b73261020bf8e9b76cf,5,Sem título,Muito bom,2017-10-15T01:00:00.000Z,2017-10-15T17:53:22.000Z
f7445b0cae0fd4d05894171adcc55931,000688fa7009b3c9fafe4950b37cbc81,5,Bom,Meu sait preferido,2018-06-19T00:00:00.000Z,2018-06-20T04:48:13.000Z
50a790149ec089ab00a4630204396f0f,00097d2ebbdd5d8921a21af2370ef764,5,Sem título,Sem comentário,2017-08-03T00:00:00.000Z,2017-08-05T23:47:13.000Z
755fdf89024021c6324864182b6657b5,0009d5c0d8e49da34e2561a67bc730f8,1,Sem título,Sem comentário,2018-03-25T00:00:00.000Z,2018-03-26T11:38:15.000Z
378e9f1b518253964f1e82f349d90b7b,000d524a3c693e342d163912ad74f156,4,Sem título,Sem comentário,2017-05-17T00:00:00.000Z,2017-05-18T11:39:45.000Z
dc6a29077e547f5fe4377f251fd1c5d0,000da26f546e5d51b6336486f57fc632,5,Sem título,"Muito satisfeita com o produto, entrega e a praticidade do site.",2017-06-03T00:00:00.000Z,2017-06-07T04:00:21.000Z
d8711cd641288f4e2e1a84dc3c6ade21,001d005e8f06c039464425cf266248e8,4,Sem título,"Inicialmente o pedido veio incompleto, passei uma msg a lannister (sem receber a resposta), recebi o restante do pedido 2 dias depois.",2017-08-01T00:00:00.000Z,2017-08-04T11:08:08.000Z
e1a73a8183d164944b0f44e1eb39ea7d,002155c71462b955916f9e3c1df7d376,5,Recomendo,Fiquei muito satisfeita.,2018-04-28T00:00:00.000Z,2018-04-30T23:48:44.000Z
7aac44d0de12f13cf2a5f9235a5a9e37,002d6d967b7d3bba097310297327864d,2,Sem título,Produto não funciona,2017-11-29T00:00:00.000Z,2017-11-29T09:31:07.000Z
207d35e4a69118f2f9422aa1949043f7,0030f939576d0188eb398c1d9c4ec1f9,4,Sem título,Sem comentário,2018-06-13T00:00:00.000Z,2018-06-14T16:41:13.000Z


root
 |-- id_pedido: string (nullable = true)
 |-- id_avaliacao: string (nullable = true)
 |-- nota_avaliacao: integer (nullable = true)
 |-- titulo_avaliacao_comentario: string (nullable = false)
 |-- mensagem_avaliacao_comentario: string (nullable = false)
 |-- data_criacao_avaliacao: timestamp (nullable = true)
 |-- data_resposta_avaliacao: timestamp (nullable = true)



In [0]:
# Salva a tabela
df_fat_avaliacoes_pedidos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{schema}.fat_avaliacoes_pedidos")

##### dim_produtos

In [0]:
# Visualização inicial
display(df_products.limit(5))
df_products.printSchema()

product_id,product_name,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,timestamp_ingestion
1e9e8ef04dbcff4541ed26657ea517e5,Perfume Premium,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,2026-04-07T20:24:39.156Z
3aa071139cb16b67ca9e5dea641aaa2f,Conjunto de Pincéis,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,2026-04-07T20:24:39.156Z
96bd76ec8810374ed1b65e291975717f,Barraca de Camping,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,2026-04-07T20:24:39.156Z
cef67bcfe19066a932b7673e239eb23d,Chupeta Premium,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,2026-04-07T20:24:39.156Z
9dc1a7de274444849c219cff195d0b71,Vassoura Mágica,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,2026-04-07T20:24:39.156Z


root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: string (nullable = true)
 |-- product_description_lenght: string (nullable = true)
 |-- product_photos_qty: string (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- product_length_cm: string (nullable = true)
 |-- product_height_cm: string (nullable = true)
 |-- product_width_cm: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
# Verificação de IDs duplicados na base original
display(
    df_products
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc(), F.col("product_id"))
)

product_id,count


In [0]:
# Deduplicação dos produtos com Window Function
window_produtos = Window.partitionBy("product_id").orderBy(F.col("timestamp_ingestion").desc())

df_products_deduplicado = (df_products.withColumn("rn", F.row_number().over(window_produtos)).filter(F.col("rn") == 1).drop("rn"))

# Verificação final de duplicidade de product_id
display(df_products_deduplicado.groupBy("product_id").count().filter(F.col("count") > 1))

product_id,count


In [0]:
# Tradução dos nomes das colunas
df_produtos_renomeado = (
    df_products_deduplicado
    .select(
        F.col("product_id").alias("id_produto"),
        F.col("product_category_name").alias("categoria_produto"),
        F.col("product_name").alias("nome_produto"),
        F.col("product_name_lenght").alias("tamanho_nome_produto"),
        F.col("product_description_lenght").alias("tamanho_descricao_produto"),
        F.col("product_photos_qty").alias("quantidade_fotos"),
        F.col("product_weight_g").alias("peso_produto_gramas"),
        F.col("product_length_cm").alias("comprimento_centimetros"),
        F.col("product_height_cm").alias("altura_centimetros"),
        F.col("product_width_cm").alias("largura_centimetros")
    )
)

In [0]:
# Tratamento dos tipos das colunas e padronização dos dados textuais
# O casting em algumas colunas para double e depois para int acontece por causa da identificação do spark das colunas como string, porém, os registros apresentam formato com casas decimais
# por exemplo: "45.0", tornando necessária a conversão para um double, e ,já que nenhum dos dados apresenta de fato um valor não inteiro, um segundo casting aplicado para inteiro é aplicado
# preferi usar try_cast para evitar que algum valor anômalo comprometa a pipeline
df_dim_produtos = (
    df_produtos_renomeado
    .withColumn("id_produto", F.trim(F.col("id_produto")))
    .withColumn("categoria_produto", F.trim(F.col("categoria_produto")))
    .withColumn("nome_produto", F.trim(F.col("nome_produto")))
    .withColumn("tamanho_nome_produto", F.expr("try_cast(trim(tamanho_nome_produto) as double)").cast("int"))
    .withColumn("tamanho_descricao_produto", F.expr("try_cast(trim(tamanho_descricao_produto) as double)").cast("int"))
    .withColumn("quantidade_fotos", F.expr("try_cast(trim(quantidade_fotos) as double)").cast("int"))
    .withColumn("peso_produto_gramas", F.expr("try_cast(trim(peso_produto_gramas) as double)").cast("int"))
    .withColumn("comprimento_centimetros", F.expr("try_cast(trim(comprimento_centimetros) as double)").cast("int"))
    .withColumn("altura_centimetros", F.expr("try_cast(trim(altura_centimetros) as double)").cast("int"))
    .withColumn("largura_centimetros", F.expr("try_cast(trim(largura_centimetros) as double)").cast("int"))
)

In [0]:
# Conversão de strings vazias para nulo nas colunas textuais
df_dim_produtos = (
    df_dim_produtos
    .withColumn("id_produto", F.when(F.col("id_produto") == "", None).otherwise(F.col("id_produto")))
    .withColumn("categoria_produto", F.when(F.col("categoria_produto") == "", None).otherwise(F.col("categoria_produto")))
    .withColumn("nome_produto", F.when(F.col("nome_produto") == "", None).otherwise(F.col("nome_produto")))
)

In [0]:
# Verificação de nulos por coluna
display(
    df_dim_produtos.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df_dim_produtos.columns
    ])
)

id_produto,categoria_produto,nome_produto,tamanho_nome_produto,tamanho_descricao_produto,quantidade_fotos,peso_produto_gramas,comprimento_centimetros,altura_centimetros,largura_centimetros
0,610,0,610,610,610,2,2,2,2


In [0]:
# Por possuir por muitas vezes linhas com 4 campos nulos, achei melhor por não inserir ou deletar nada. Achei melhor sinalizar, uma vez que isso pode ser um problema da origem
df_dim_produtos = (
    df_dim_produtos
    .withColumn(
        "flag_qualidade",
        F.when(
            F.col("id_produto").isNull(),
            "id_produto_nulo"
        ).when(
            F.col("categoria_produto").isNull() |
            F.col("nome_produto").isNull() |
            F.col("tamanho_nome_produto").isNull() |
            F.col("tamanho_descricao_produto").isNull() |
            F.col("quantidade_fotos").isNull(),
            "cadastro_produto_incompleto"
        ).when(
            F.col("peso_produto_gramas").isNull() |
            F.col("comprimento_centimetros").isNull() |
            F.col("altura_centimetros").isNull() |
            F.col("largura_centimetros").isNull(),
            "dimensao_fisica_incompleta"
        ).otherwise(None)
    )
)

In [0]:
# Visualização dos tipos de problema capturados pela flag de qualidade
display(df_dim_produtos.groupBy("flag_qualidade").count())

flag_qualidade,count
null,32340
cadastro_produto_incompleto,610
dimensao_fisica_incompleta,1


In [0]:
# Tratamento de nulos nas colunas textuais mais relevantes para consumo analítico
# Optei por tratar apenas os campos textuais principais da dimensão.

df_dim_produtos = (
    df_dim_produtos
    .fillna({
        "categoria_produto": "SEM CATEGORIA",
        "nome_produto": "SEM NOME"
    })
)

In [0]:
# Visualização das linhas "problemáticas" identificadas
display(df_dim_produtos.filter(F.col("flag_qualidade").isNotNull()).limit(10))

id_produto,categoria_produto,nome_produto,tamanho_nome_produto,tamanho_descricao_produto,quantidade_fotos,peso_produto_gramas,comprimento_centimetros,altura_centimetros,largura_centimetros,flag_qualidade
0082684bb4a60a862baaf7a60a5845ed,SEM CATEGORIA,Kit Variado Branco,null,null,null,500,24,4,15,cadastro_produto_incompleto
00ab8a8b9fe219511dc3f178c6d79698,SEM CATEGORIA,Produto Genérico Master,null,null,null,2100,50,30,30,cadastro_produto_incompleto
00d62b338366db4c4aec8547ea8f928e,SEM CATEGORIA,Acessório Padrão,null,null,null,400,24,4,15,cadastro_produto_incompleto
0103863bf3441460142ec23c74388e4c,SEM CATEGORIA,Item Básico Avançado,null,null,null,200,16,2,11,cadastro_produto_incompleto
014fcf6bd5cd4c7ee29fb3bb618c445e,SEM CATEGORIA,Peça de Reposição,null,null,null,7000,55,30,45,cadastro_produto_incompleto
017457b0013d01d5a5a4a020ed1f14b9,SEM CATEGORIA,Peça de Reposição Preto,null,null,null,2200,28,20,14,cadastro_produto_incompleto
0292b46c30348496b1be04eeb440bdb5,SEM CATEGORIA,Kit Variado,null,null,null,400,38,22,15,cadastro_produto_incompleto
033fe931444d66d788d53aca5b7b22d5,SEM CATEGORIA,Produto Genérico Dourado,null,null,null,200,16,19,11,cadastro_produto_incompleto
036a6272d538ae862d248339c4c9adeb,SEM CATEGORIA,Equipamento Utilitário Cinza,null,null,null,1800,20,12,25,cadastro_produto_incompleto
03706dca83513062fe74967a71b5fc78,SEM CATEGORIA,Item Básico,null,null,null,400,25,8,11,cadastro_produto_incompleto


In [0]:
# Visualização final
display(df_dim_produtos.limit(10))
df_dim_produtos.printSchema()

id_produto,categoria_produto,nome_produto,tamanho_nome_produto,tamanho_descricao_produto,quantidade_fotos,peso_produto_gramas,comprimento_centimetros,altura_centimetros,largura_centimetros,flag_qualidade
00066f42aeeb9f3007548bb9d3f33c38,perfumaria,Loção Corporal Preto,53,596,6,300,20,16,16,null
00088930e925c41fd95ebfe695fd2655,automotivo,Central Multimídia Avançado,56,752,4,1225,55,10,26,null
0009406fd7479715e4bef61dd91f2462,cama_mesa_banho,Toalha de Banho Premium,50,266,2,300,45,15,35,null
000b8f95fcb9e0096488278317764d19,utilidades_domesticas,Tábua de Corte,25,364,3,550,19,24,12,null
000d9be29b5207b54e86aa1b1ac54872,relogios_presentes,Relógio Analógico Dourado,48,613,4,250,22,11,15,null
0011c512eb256aa0dbbb544d8dffcf6e,automotivo,Bateria Automotiva Luxo,58,177,1,100,16,15,16,null
00126f27c813603687e6ce486d909d01,cool_stuff,Quadro Nerd Master,42,2461,1,700,25,5,15,null
001795ec6f1b187d37335e1c4704762e,consoles_games,Acessório Padrão Luxo,53,274,1,600,30,20,20,null
001b237c0e9bb435f2e54071129237e9,cama_mesa_banho,Edredom,42,253,1,6000,40,4,30,null
001b72dfd63e9833e8c02742adf472e3,moveis_decoracao,Mesa de Jantar,45,520,3,600,26,8,22,null


root
 |-- id_produto: string (nullable = true)
 |-- categoria_produto: string (nullable = false)
 |-- nome_produto: string (nullable = false)
 |-- tamanho_nome_produto: integer (nullable = true)
 |-- tamanho_descricao_produto: integer (nullable = true)
 |-- quantidade_fotos: integer (nullable = true)
 |-- peso_produto_gramas: integer (nullable = true)
 |-- comprimento_centimetros: integer (nullable = true)
 |-- altura_centimetros: integer (nullable = true)
 |-- largura_centimetros: integer (nullable = true)
 |-- flag_qualidade: string (nullable = true)



In [0]:
# Salva a tabela
df_dim_produtos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{schema}.dim_produtos")

##### dim_vendedores

In [0]:
# Visualização inicial
display(df_sellers.limit(5))
df_sellers.printSchema()

seller_id,seller_zip_code_prefix,seller_city,seller_state,seller_name,seller_registration_date,timestamp_ingestion
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,Dr. Pedro Miguel Vasconcelos,2018-06-25,2026-04-07T20:24:41.730Z
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,Gabriela da Cruz,2018-09-21,2026-04-07T20:24:41.730Z
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,Luigi Viana,2018-04-03,2026-04-07T20:24:41.730Z
c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP,Beatriz Brito,2018-12-20,2026-04-07T20:24:41.730Z
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,Gabrielly Pastor,2017-08-24,2026-04-07T20:24:41.730Z


root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- seller_name: string (nullable = true)
 |-- seller_registration_date: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
# Verificação de IDs duplicados na base original
display(df_sellers.groupBy("seller_id").count().filter(F.col("count") > 1).orderBy(F.col("count").desc(), F.col("seller_id")))

seller_id,count


In [0]:
# Deduplicação dos vendedores com Window Function
window_vendedores = Window.partitionBy("seller_id").orderBy(F.col("timestamp_ingestion").desc())

df_sellers_deduplicado = (df_sellers.withColumn("rn", F.row_number().over(window_vendedores)).filter(F.col("rn") == 1).drop("rn"))

# Verificação final de duplicidade após a deduplicação
display(df_sellers_deduplicado.groupBy("seller_id").count().filter(F.col("count") > 1))

seller_id,count


In [0]:
# Tradução dos nomes das colunas
df_vendedores_renomeado = (
    df_sellers_deduplicado
    .select(
        F.col("seller_id").alias("id_vendedor"),
        F.col("seller_name").alias("nome_vendedor"),
        F.col("seller_zip_code_prefix").alias("prefixo_cep"),
        F.col("seller_city").alias("cidade"),
        F.col("seller_state").alias("estado"),
        F.col("seller_registration_date").alias("data_cadastro_vendedor")
    )
)

In [0]:
# Tratamento dos tipos e padronização das strings
# Decidi tratar cep como string por ser um dado que as vezes é inserido de forma diferente ("alguns inserem só os números e outros inserem com o "-"), além de que no 
# tipo int, se o CEP começar com '0', provavelmente ele será desconsiderado por ser um zero à esquerda e não agregar valor numérico
df_dim_vendedores = (
    df_vendedores_renomeado
    .withColumn("id_vendedor", F.trim(F.col("id_vendedor")))
    .withColumn("nome_vendedor", F.trim(F.col("nome_vendedor")))
    .withColumn("cidade", F.upper(F.trim(F.col("cidade"))))
    .withColumn("estado", F.upper(F.trim(F.col("estado"))))
    .withColumn("prefixo_cep", F.trim(F.col("prefixo_cep").cast("string")))
    .withColumn("data_cadastro_vendedor", F.to_timestamp(F.col("data_cadastro_vendedor")))
)

In [0]:
# Conversão de strings vazias para nulo nas colunas textuais
df_dim_vendedores = (
    df_dim_vendedores
    .withColumn("id_vendedor", F.when(F.col("id_vendedor") == "", None).otherwise(F.col("id_vendedor")))
    .withColumn("nome_vendedor", F.when(F.col("nome_vendedor") == "", None).otherwise(F.col("nome_vendedor")))
    .withColumn("cidade", F.when(F.col("cidade") == "", None).otherwise(F.col("cidade")))
    .withColumn("estado", F.when(F.col("estado") == "", None).otherwise(F.col("estado")))
    .withColumn("prefixo_cep", F.when(F.col("prefixo_cep") == "", None).otherwise(F.col("prefixo_cep")))
)

In [0]:
# Verificação de nulos por coluna
display(
    df_dim_vendedores.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df_dim_vendedores.columns
    ])
)

id_vendedor,nome_vendedor,prefixo_cep,cidade,estado,data_cadastro_vendedor
0,0,0,0,0,0


In [0]:
# Tratamento de nulos em colunas textuais relevantes para consumo analítico
df_dim_vendedores = (
    df_dim_vendedores
    .fillna({
        "nome_vendedor": "SEM NOME",
        "cidade": "NAO INFORMADO",
        "estado": "SEM ESTADO"
    })
)

In [0]:
# Visualização final
display(df_dim_vendedores.limit(10))
df_dim_vendedores.printSchema()

id_vendedor,nome_vendedor,prefixo_cep,cidade,estado,data_cadastro_vendedor
0015a82c2db000af6aaaf3ae2ecb0532,Amanda Sá,9080,SANTO ANDRE,SP,2018-09-29T00:00:00.000Z
001cca7ae9ae17fb1caed9dfb1094831,Vinicius Nogueira,29156,CARIACICA,ES,2017-11-09T00:00:00.000Z
001e6ad469a905060d959994f1b41e4f,Ana Clara Moreira,24754,SAO GONCALO,RJ,2018-11-16T00:00:00.000Z
002100f778ceb8431b7a1020ff7ab48f,Srta. Emanuella Rezende,14405,FRANCA,SP,2017-04-17T00:00:00.000Z
003554e2dce176b5555353e4f3555ac8,Rebeca Costa,74565,GOIANIA,GO,2017-07-20T00:00:00.000Z
004c9cd9d87a3c30c522c48c4fc07416,Aylla da Rocha,14940,IBITINGA,SP,2017-05-22T00:00:00.000Z
00720abe85ba0859807595bbf045a33b,Bruna Barbosa,7070,GUARULHOS,SP,2017-07-25T00:00:00.000Z
00ab3eff1b5192e5f1a63bcecfee11c8,Bella da Luz,4164,SAO PAULO,SP,2017-11-18T00:00:00.000Z
00d8b143d12632bad99c0ad66ad52825,Luiz Miguel Moura,30170,BELO HORIZONTE,MG,2018-11-19T00:00:00.000Z
00ee68308b45bc5e2660cd833c3f81cc,Thiago da Conceição,3333,SAO PAULO,SP,2019-01-02T00:00:00.000Z


root
 |-- id_vendedor: string (nullable = true)
 |-- nome_vendedor: string (nullable = false)
 |-- prefixo_cep: string (nullable = true)
 |-- cidade: string (nullable = false)
 |-- estado: string (nullable = false)
 |-- data_cadastro_vendedor: timestamp (nullable = true)



In [0]:
# Salva a tabela
df_dim_vendedores.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{schema}.dim_vendedores")

##### dim_categoria_produtos_traducao

In [0]:
# Visualização inicial
display(df_category_translation.limit(5))
df_category_translation.printSchema()

product_category_name,product_category_name_english,timestamp_ingestion
beleza_saude,health_beauty,2026-04-07T20:24:44.243Z
informatica_acessorios,computers_accessories,2026-04-07T20:24:44.243Z
automotivo,auto,2026-04-07T20:24:44.243Z
cama_mesa_banho,bed_bath_table,2026-04-07T20:24:44.243Z
moveis_decoracao,furniture_decor,2026-04-07T20:24:44.243Z


root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
# Remoção de duplicatas
df_categoria_traducao_deduplicado = df_category_translation.dropDuplicates()

In [0]:
# Tradução dos nomes das colunas
df_categoria_traducao_renomeado = (
    df_categoria_traducao_deduplicado
    .select(
        F.col("product_category_name").alias("nome_produto_pt"),
        F.col("product_category_name_english").alias("nome_produto_en")
    )
)

In [0]:
# Padronização das strings
df_dim_categoria_produtos_traducao = (
    df_categoria_traducao_renomeado
    .withColumn("nome_produto_pt", F.trim(F.col("nome_produto_pt")))
    .withColumn("nome_produto_en", F.trim(F.col("nome_produto_en")))
)

In [0]:
# Conversão de strings vazias para nulo
# Por ser uma tabela mais simples, decidi não fazer um tratamento específico para os nulos
df_dim_categoria_produtos_traducao = (
    df_dim_categoria_produtos_traducao
    .withColumn("nome_produto_pt", F.when(F.col("nome_produto_pt") == "", None).otherwise(F.col("nome_produto_pt")))
    .withColumn("nome_produto_en", F.when(F.col("nome_produto_en") == "", None).otherwise(F.col("nome_produto_en")))
)

In [0]:
# Verificação de nulos por coluna
display(df_dim_categoria_produtos_traducao.select([F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df_dim_categoria_produtos_traducao.columns
    ])
)

nome_produto_pt,nome_produto_en
0,0


In [0]:
# Visualização final
display(df_dim_categoria_produtos_traducao.limit(10))
df_dim_categoria_produtos_traducao.printSchema()

nome_produto_pt,nome_produto_en
beleza_saude,health_beauty
informatica_acessorios,computers_accessories
automotivo,auto
cama_mesa_banho,bed_bath_table
moveis_decoracao,furniture_decor
esporte_lazer,sports_leisure
perfumaria,perfumery
utilidades_domesticas,housewares
telefonia,telephony
relogios_presentes,watches_gifts


root
 |-- nome_produto_pt: string (nullable = true)
 |-- nome_produto_en: string (nullable = true)



In [0]:
# Salva a tabela
df_dim_categoria_produtos_traducao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{schema}.dim_categoria_produtos_traducao")

##### dim_cotacao_dolar

In [0]:
# Visualização inicial
display(df_cotacao_dolar.limit(5))
df_cotacao_dolar.printSchema()

cotacaoCompra,dataHoraCotacao,timestamp_ingestion
3.9907,2016-03-01 13:10:04.662,2026-04-07T20:24:51.685Z
3.911,2016-03-02 13:12:16.932,2026-04-07T20:24:51.685Z
3.8498,2016-03-03 13:04:27.128,2026-04-07T20:24:51.685Z
3.7182,2016-03-04 13:06:46.766,2026-04-07T20:24:51.685Z
3.7708,2016-03-07 13:02:47.751,2026-04-07T20:24:51.685Z


root
 |-- cotacaoCompra: double (nullable = true)
 |-- dataHoraCotacao: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
# Tradução dos nomes das colunas
df_cotacao_dolar_renomeado = (
    df_cotacao_dolar
    .select(
        F.col("cotacaoCompra").alias("cotacao_compra"),
        F.col("dataHoraCotacao").alias("data_hora_cotacao")
    )
)

In [0]:
# Tratamento inicial
# Convertemos a coluna de data/hora para timestamp e derivamos a data da cotação,
# que será a granularidade final da dimensão.
df_cotacao_tratado = (
    df_cotacao_dolar_renomeado
    .withColumn("data_hora_cotacao", F.to_timestamp(F.col("data_hora_cotacao")))
    .withColumn("data_cotacao", F.to_date(F.col("data_hora_cotacao")))
    .withColumn("cotacao_compra", F.col("cotacao_compra").cast("double"))
)

display(df_cotacao_tratado.limit(5))

cotacao_compra,data_hora_cotacao,data_cotacao
3.9907,2016-03-01T13:10:04.662Z,2016-03-01
3.911,2016-03-02T13:12:16.932Z,2016-03-02
3.8498,2016-03-03T13:04:27.128Z,2016-03-03
3.7182,2016-03-04T13:06:46.766Z,2016-03-04
3.7708,2016-03-07T13:02:47.751Z,2016-03-07


In [0]:
# Seleção das colunas úteis para a dimensão final
df_cotacao_base = (
    df_cotacao_tratado
    .select("data_cotacao", "cotacao_compra")
)

display(df_cotacao_base.limit(5))

data_cotacao,cotacao_compra
2016-03-01,3.9907
2016-03-02,3.911
2016-03-03,3.8498
2016-03-04,3.7182
2016-03-07,3.7708


In [0]:
# Criação de calendário contínuo entre a menor e a maior data disponível
intervalo_datas = df_cotacao_base.select(
    F.min("data_cotacao").alias("data_min"),
    F.max("data_cotacao").alias("data_max")
)

df_calendario = (
    intervalo_datas
    .select(F.explode(F.sequence(F.col("data_min"), F.col("data_max"))).alias("data_cotacao"))
)

# Junta o calendário com a base de cotação
df_cotacao_calendario = (
    df_calendario
    .join(df_cotacao_base, on="data_cotacao", how="left")
)

In [0]:
# Remoção de datas repetidas, priorizando registros com cotação preenchida
window_datas_repetidas = (
    Window
    .partitionBy("data_cotacao")
    .orderBy(F.col("cotacao_compra").isNull().cast("int").asc())
)

df_cotacao_calendario = (
    df_cotacao_calendario
    .withColumn("rn", F.row_number().over(window_datas_repetidas))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

In [0]:
# Preenchimento dos dias sem cotação com a última cotação válida anterior
window_preenchimento = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_dim_cotacao_dolar = (
    df_cotacao_calendario
    .withColumn(
        "cotacao_compra",
        F.last("cotacao_compra", ignorenulls=True).over(window_preenchimento)
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# Visualização final
display(df_dim_cotacao_dolar.orderBy("data_cotacao").limit(10))
df_dim_cotacao_dolar.printSchema()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_cotacao,cotacao_compra
2016-03-01,3.9907
2016-03-02,3.911
2016-03-03,3.8498
2016-03-04,3.7182
2016-03-05,3.7182
2016-03-06,3.7182
2016-03-07,3.7708
2016-03-08,3.7807
2016-03-09,3.7031
2016-03-10,3.6694


root
 |-- data_cotacao: date (nullable = false)
 |-- cotacao_compra: double (nullable = true)



In [0]:
# Salva a tabela
df_dim_cotacao_dolar.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{schema}.dim_cotacao_dolar")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


##### fat_pedido_total

In [0]:
# Consolidação dos pagamentos de cada pedido (soma das "parcelas")
df_pagamentos_por_pedido = (
    df_fat_pagamentos_pedidos
    .groupBy("id_pedido")
    .agg(
        F.round(F.sum("valor_pagamento_BRL"), 2).alias("valor_total_pago_brl")
    )
)

# Apenas visualização do df
display(df_pagamentos_por_pedido.limit(10))
df_pagamentos_por_pedido.printSchema()

id_pedido,valor_total_pago_brl
00010242fe8c5a6d1ba2dd792cb16214,72.19
00018f77f2f0320c557190d7a144bdd3,259.83
000229ec398224ef6ca0657da4fc703e,216.87
00024acbcdf0a6daa1e931b038114c75,25.78
00042b26cf59d7ce69dfabb4e55b4fd9,218.04
00048cc3ae777c65dbb7d2a0634bc1ea,34.59
00054e8431b9d7675808bcb819fb4a32,31.75
000576fe39319847cbb9d288c5617fa6,880.75
0005a1a1728c9d785b8e2b08b904576c,157.60
0005f50442cb953dcd1d21e1fb923495,65.39


root
 |-- id_pedido: string (nullable = true)
 |-- valor_total_pago_brl: decimal(21,2) (nullable = true)



In [0]:
# Construção da tabela final fat_pedido_total
df_fat_pedido_total = (
    df_fat_pedidos
    .withColumn("data_pedido", F.to_date(F.col("data_compra"))) # Coluna para ajudar na conversão de real para dolar
    .join(df_pagamentos_por_pedido, on="id_pedido", how="left") # Junção dos pedidos com os seus valores pagos (em BRL)
    .join(
        df_dim_cotacao_dolar.select("data_cotacao", "cotacao_compra"),
        F.col("data_pedido") == F.col("data_cotacao"),
        how="left"
    ) # Junção da tabela que está sendo construída com a tabela de cotação do dólar
    .withColumn(
        "valor_total_pago_brl",
        F.round(F.col("valor_total_pago_brl"), 2) # Arredondamento para 2 casas decimais
    )
    .withColumn(
        "valor_total_pago_usd",
        F.round(F.col("valor_total_pago_brl") / F.col("cotacao_compra"), 2) # Cálculo de conversão pro dólar com arredondamento de 2 casas decimais
    )
    .select(
        "id_pedido",
        "id_consumidor",
        "status",
        "valor_total_pago_brl",
        "valor_total_pago_usd",
        "data_pedido"
    ) # Seleção das colunas importante e exclusão das auxiliares
)

# Apenas visualização do df
display(df_fat_pedido_total.limit(10))
df_fat_pedido_total.printSchema()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_pedido,id_consumidor,status,valor_total_pago_brl,valor_total_pago_usd,data_pedido
000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,entregue,216.87,67.37,2018-01-14
00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,entregue,218.04,69.81,2017-02-04
0005f50442cb953dcd1d21e1fb923495,351d3cb2cee3c7fd0af6616c82df21d3,entregue,65.39,16.75,2018-07-02
00063b381e2406b52ad429470734ebd5,6a899e55865de6549a58d2c6845e5604,entregue,57.98,15.6,2018-07-27
0009792311464db532ff765bf7b182ae,2a30c97668e81df7c17a8b14447aeeba,entregue,127.55,32.87,2018-08-14
001021efaa8636c29475e7734483457d,2dfbf74859104caf100df3720a1d833d,entregue,64.10,19.8,2018-02-27
0010b2e5201cc5f1ae7e9c6cc8f5bd00,57ef317d4818cb42680fc9dfd13867ce,entregue,65.50,21.23,2017-09-11
001dbc16dc51075e987543d23a0507c7,698a74f33469466fa4172e829505d1c6,entregue,87.90,27.83,2017-01-28
0020a222f55eb79a372d0efee3cca688,0c45155afd8ff99622c40824057f9b34,entregue,45.09,14.1,2017-08-15
00254baeb6c932b0a8aeead91fbd02b5,ce0421a97232c2a1194cdb66cd3ebb9d,entregue,193.01,53.94,2018-05-08


root
 |-- id_pedido: string (nullable = true)
 |-- id_consumidor: string (nullable = true)
 |-- status: string (nullable = false)
 |-- valor_total_pago_brl: decimal(22,2) (nullable = true)
 |-- valor_total_pago_usd: double (nullable = true)
 |-- data_pedido: date (nullable = true)



In [0]:
# Verificação de valores nulos 
# Foi registrado como nulo um único pedido que não tem registro em pagamentos
display(
    df_fat_pedido_total.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df_fat_pedido_total.columns
    ])
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_pedido,id_consumidor,status,valor_total_pago_brl,valor_total_pago_usd,data_pedido
0,0,0,1,1,0


In [0]:
# Verificação de possíveis pedidos duplicados
display(
    df_fat_pedido_total
    .groupBy("id_pedido")
    .count()
    .filter(F.col("count") > 1)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_pedido,count


In [0]:
# Salvando a tabela final da camada Silver
df_fat_pedido_total.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{schema}.fat_pedido_total")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


#### Etapa de otimização física
___


In [0]:
# Otimização física das principais tabelas fato da camada Silver
# Confesso que fiquei um pouco confuso nessa parte de utilizar data_pedido, já que ela só surge na tabela fat_pedido_total
# Devido a essa confusão, usei ela na tabela que ela existia, mas, nas outras usei a medida temporária que mais tem impacto na tabela
spark.sql(f"""
OPTIMIZE {catalogo}.{schema}.fat_pedidos
ZORDER BY (id_pedido, data_compra)
""")

spark.sql(f"""
OPTIMIZE {catalogo}.{schema}.fat_itens_pedidos
ZORDER BY (id_pedido)
""")

spark.sql(f"""
OPTIMIZE {catalogo}.{schema}.fat_pagamentos_pedidos
ZORDER BY (id_pedido)
""")

spark.sql(f"""
OPTIMIZE {catalogo}.{schema}.fat_avaliacoes_pedidos
ZORDER BY (id_pedido)
""")

spark.sql(f"""
OPTIMIZE {catalogo}.{schema}.fat_pedido_total
ZORDER BY (id_pedido, data_pedido)
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,